# 21 - Budget-matched fine-tune test (Part A) + era split where hospitals really differ (Part B)

**Pre-registration:** `docs/2026-10-29-budget-and-era-preregistration.md` (read it first; criteria fixed before this runs).
**Background / offline evidence:** `docs/2026-10-29-label-heterogeneity-and-fingerprints.md`.

**Part A — is FedAvg + fine-tune's lead just 8 extra epochs?** Louvain (3 hospitals) and organism-10. Per seed:
`long` = FedAvg **24 rounds** (x4 local epochs) + local-only **104 epochs** + fine-tune 8, and `ft8` = FedAvg 8 rounds + fine-tune 8
(the notebook 16/17 arm, rerun on this machine). Every smaller budget (local-only at 32/40/64 epochs, FedAvg at 8/10/16 rounds)
is read out of `long` afterwards (`federated/budget.py`), so nothing is retrained.

**Part B — can similarity weighting help when hospitals differ in P(resistant | x)?** `era_cluster_split`: 10 hospitals in
2 groups of 5; each group's patients come from one half of the 2008-2023 timeline (testing panels and breakpoints changed
over time, and the model has no time feature). Arms, all ending with the same 8-epoch fine-tune (budget-matched at 40 epochs):
FedAvg (+ local-only 40), v3 with the current topology fingerprint, v3 with the **cross-antibiogram** fingerprint, its
shuffled control, an **oracle** that averages exactly within the true era group, and SimFT.

**Runtime (rough, Colab T4):** Part A ~2-3 h, Part B ~1.5-2 h. Two accounts: set `RUN_PART` / `TAG` in cell 4
(A on one, B on the other); cell 10 merges. Everything is checkpointed after every seed and resumes.

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/budget-and-era'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx scipy
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Sanity: data resolves + this branch has the notebook-21 modules
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import budget_runs as BR, budget as B, final_protocol as P
from amr_fed.partition import era_cluster_split
from amr_fed.label_fingerprints import CANDIDATES
print('imports OK | long run:', BR.LONG_ROUNDS, 'rounds, local-only', BR.LONG_LOCAL, 'epochs | Part B arms:', list(BR.PART_B_ARMS))

In [ ]:
# 3) Load cohort once + smoke-check the three splits
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split, organism_community
from amr_fed.split_diagnostics import heterogeneity_vs_null, history_strata
df = load_cohort_frame()
print('rows:', f'{len(df):,}')
SEEDS = (42, 43, 44, 45, 46)
SPLITS = {
    'louvain': (lambda d, s: louvain_split(d, n_clients=3, seed=s), 3),
    'org10':   (lambda d, s: organism_community(d, n_clients=10, seed=s), 10),
}
ERA_SPLIT = lambda d, s: era_cluster_split(d, n_clients=10, n_eras=2, seed=s)
for name, (fn, k) in SPLITS.items():
    a = fn(df, 42); assert a.nunique() == k, f'{name}: {a.nunique()} hospitals'
a = ERA_SPLIT(df, 42)
assert a.nunique() == 10 and a.index.is_unique and len(a) == df[PK].nunique()
print('era split sizes:', a.value_counts().sort_index().tolist())
h = heterogeneity_vs_null(df, a, n_null=2, strata=history_strata(df))
print(f"era split concept shift: {h['concept']:.2f} vs random {h['concept_null']:.2f} (ratio {h['concept_ratio']:.2f}; expected ~1.5)")
print('SMOKE PASS')

In [ ]:
# 4) What this session runs. Two accounts: RUN_PART='A', TAG='A' on one and RUN_PART='B', TAG='B' on the other.
import json, ray
os.environ.pop('AMR_RNG_PROBE', None)
RUN_PART = 'all'          # 'A', 'B' or 'all'
TAG = 'all'
RUN_SETTINGS = ['louvain', 'org10']   # Part A settings (louvain is cheaper: run it first)
BASE = dict(seeds=SEEDS, df=df, rounds=8, local_epochs=BR.LOCAL_EPOCHS, local_only_epochs=40, hidden=128)
RES_DIR = SRC.parent / 'results'; RES_DIR.mkdir(exist_ok=True)

def paths(stem):
    local = RES_DIR / f'{stem}_{TAG}.json'
    mirror = Path(f'/content/drive/MyDrive/{stem}_{TAG}.json') if ON_COLAB else None
    return local, mirror

def load(stem):
    local, mirror = paths(stem)
    src = local if local.exists() else (mirror if mirror and mirror.exists() else None)
    return json.load(open(src)) if src else {}

In [ ]:
# 5) PART A: long + ft8 arms per setting (resume-safe)
if RUN_PART in ('A', 'all'):
    for setting in RUN_SETTINGS:
        fn, k = SPLITS[setting]
        stem = f'amr_budget_{setting}'
        res = load(stem)
        kw = dict(BASE, partition_fn=fn, n_clients=k)
        res = BR.run_part(res, BR.PART_A_ARMS, prefix=f'budget/{setting}', kw=kw,
                          checkpoint=lambda r, stem=stem: P.save_json(r, *paths(stem)), ray_shutdown=ray.shutdown)
        P.save_json(res, *paths(stem))
        print(setting, 'arms done:', [a for a in res if not a.startswith('_')])

In [ ]:
# 6) PART A report: every budget, paired comparisons, plateau check, determinism check
for setting in RUN_SETTINGS:
    res = load(f'amr_budget_{setting}')
    if all(a in res for a in BR.PART_A_ARMS):
        print(f'\n################ {setting} ################')
        print(BR.part_a_report(res))
    else:
        print(setting, ': not finished')

In [ ]:
# 7) PART B: era split, six arms (resume-safe)
if RUN_PART in ('B', 'all'):
    stem = 'amr_era10'
    res_b = load(stem)
    kw = dict(BASE, partition_fn=ERA_SPLIT, n_clients=10)
    res_b = BR.run_part(res_b, BR.PART_B_ARMS, prefix='era10', kw=kw,
                        checkpoint=lambda r: P.save_json(r, *paths(stem)), ray_shutdown=ray.shutdown)
    P.save_json(res_b, *paths(stem))
    print('arms done:', [a for a in res_b if not a.startswith('_')])

In [ ]:
# 8) PART B report: pre-registered comparisons + where each personalized arm put its weight
res_b = load('amr_era10')
print(BR.part_b_report(res_b))
groups = {s: BR.era_split(df, s)[1] for s in SEEDS}
print('weights (mean over hospitals and seeds):')
print(BR.part_b_weights(res_b, groups))

In [ ]:
# 9) Offline fingerprint test on the era split: does each fingerprint predict SimFT's measured transfer gaps?
if 'simft' in res_b:
    frames = {s: BR.era_train_frames(df, s) for s in res_b['simft']['seeds']}
    print(BR.offline_fingerprint_test(res_b, frames))
else:
    print('simft arm not finished')

In [ ]:
# 10) Merge the two accounts' files (put both in results/ or Drive first), then re-run cells 6, 8, 9 with TAG='merged'
import glob
for stem in ['amr_budget_louvain', 'amr_budget_org10', 'amr_era10']:
    parts = sorted(glob.glob(str(RES_DIR / f'{stem}_[!m]*.json')))
    if ON_COLAB: parts += sorted(glob.glob(f'/content/drive/MyDrive/{stem}_[!m]*.json'))
    if parts:
        merged = P.merge_results(parts)
        P.save_json(merged, RES_DIR / f'{stem}_merged.json')
        print(stem, 'merged arms:', [a for a in merged if not a.startswith('_')], 'from', parts)
# TAG = 'merged'   # then re-run cells 6, 8 and 9